In [62]:
import os
import glob
from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages import HumanMessage, SystemMessage


In [63]:
MODEL = "claude-sonnet-4-5-20250929"
DB_NAME = "../../vector_db"

In [64]:
content =''
with open("../../knowledge-base/dhurandhar-RAG.md", 'r', encoding='utf-8') as f:
    content += f.read()
    content += "\n\n"

print(f"Total characters in knowledge base: {len(content):,}")

Total characters in knowledge base: 7,246


In [65]:
# Load in everything in the knowledgebase using LangChain's loaders

folders = glob.glob("../../knowledge-base")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

Loaded 1 documents


In [66]:
# Divide into chunks using the RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 10 chunks
First chunk:

page_content='**Dhurandhar: The Revenge**

Theatrical release poster
Directed by	Aditya Dhar
Written by	Aditya Dhar
Additional screenplay	Ojas Gautam
Shivkumar V. Panicker
Produced by	
Aditya Dhar
Jyoti Deshpande
Lokesh Dhar

**Starring**  
Ranveer Singh
Arjun Rampal
Sanjay Dutt
R. Madhavan
Sara Arjun
Rakesh Bedi
Gaurav Gera
Danish Pandor

Cinematography	Vikash Nowlakha.    
Edited by	Shivkumar V. Panicker.  
Music by	Shashwat Sachdev. 

**Production companies**
Jio Studios     
B62 Studios     
Distributed by	Jio Studios     

Release date	
19 March 2026

Running time	229 minutes[1]      
Country	India       
Language	Hindi       
Budget	₹250–255 crore (combined with part 1)[2]        
Box office	₹1,778.25 crore[3]' metadata={'source': '../../knowledge-base/dhurandhar-RAG.md', 'doc_type': 'knowledge-base'}


In [67]:
# Pick an embedding model

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

if os.path.exists(DB_NAME):
    Chroma(persist_directory=DB_NAME, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=DB_NAME)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vectorstore created with 10 documents


In [68]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory='../../vector_db', embedding_function=embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [69]:
retriever = vectorstore.as_retriever()
llm = ChatAnthropic(temperature=0, model_name=MODEL)

In [70]:
retriever.invoke("How did SP aslam die?")

[Document(id='d5a0ebad-cd5d-4378-a442-c6b0796e8c6e', metadata={'source': '../../knowledge-base/dhurandhar-RAG.md', 'doc_type': 'knowledge-base'}, page_content="As suspicion grows, SP Chaudhary Aslam begins investigating Hamza but is killed in a suicide attack orchestrated by Hamza through the Balochistan United Force (BUF). Yalina uncovers Hamza's true identity and confronts him, but agrees to remain silent for the sake of their son, Zayan. Sanyal grants Hamza autonomy to dismantle the remaining terror network, leading him to assassinate several key operatives including terror financier Javed Khanani and IC 814 hijacker Zahoor Mistry. Meanwhile, Aslam's second-in-command Omar uncovers Hamza's true identity by extorting Yalina after holding Zayan at gunpoint, and informs Major Iqbal."),
 Document(id='0cbc1c36-ab94-4b73-9ef2-57b0e01cf480', metadata={'doc_type': 'knowledge-base', 'source': '../../knowledge-base/dhurandhar-RAG.md'}, page_content='**Plot**        \nThe film is divided into 

In [71]:
SYSTEM_PROMPT_TEMPLATE = """
You are a movie wiki chatbot.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [72]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [73]:
answer_question("How did SP aslam die?",[])
answer_question("Who directed dhurandhar",[])

'According to the information provided, **Aditya Dhar** directed Dhurandhar: The Revenge. He also wrote the film and served as one of its producers alongside Lokesh Dhar and Jyoti Deshpande.'